## 7 - Baseline

Entraînement d'un modèle trivial servant de point de comparaison pour les modèles plus complexes.

- La baseline fixe un seuil minimal : tout modèle plus complexe doit la battre pour justifier son coût.
- Sans point de comparaison, un score MAE ou RMSE n'est pas interprétable en soi.
- `DummyRegressor` : stratégie à choisir entre moyenne et médiane selon la distribution de la cible.

La distribution de `SalePrice` est asymétrique à droite. Prédire la moyenne donnerait une MAE artificiellement élevée. La médiane est la constante mathématique qui minimise la MAE, c'est donc notre stratégie de baseline.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import sys
sys.path.append("..")
from src.preprocessing import regrouper_categories_rares, traitement_valeurs_incohérentes, zero_vers_nan, Transformation_binaire, remplacer_au_dessus_seuil, fusionner_categories, extraire_dates, imputation_mediane_a_nan

# 1. Chargement
df = pd.read_csv("../data/raw/bluebook-for-bulldozers/TrainAndValid.csv", low_memory=False)
df["saledate"] = pd.to_datetime(df["saledate"])

# 2. Nettoyage global
taux_nan = df.isnull().mean()
df = df.drop(columns=taux_nan[taux_nan > 0.7].index)
df = traitement_valeurs_incohérentes(df, "YearMade")
df = zero_vers_nan(df, "MachineHoursCurrentMeter")
df = Transformation_binaire(df, "MachineHoursCurrentMeter", "hours_reported")
df = remplacer_au_dessus_seuil(df, "MachineHoursCurrentMeter", 40000)

# 3. Split temporel
train = df[df["saledate"] < "2012-01-01"].copy()
test = df[df["saledate"] >= "2012-01-01"].copy()

# 4. Transformations sur train/test séparés (pour éviter la fuite de données)
train, test = regrouper_categories_rares(train, test, "Hydraulics", 500)
train, test = regrouper_categories_rares(train, test, "fiProductClassDesc", 500)

mapping_enclosure = {"NO ROPS": "OROPS", "EROPS AC": "EROPS w AC", "None or Unspecified": np.nan}
train = fusionner_categories(train, "Enclosure", mapping_enclosure)
test = fusionner_categories(test, "Enclosure", mapping_enclosure)

In [2]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

# 1. Séparation X et y (en excluant SalePrice et MachineID)
X_train = train.drop(columns=["SalePrice", "MachineID"])
y_train = train["SalePrice"]

X_test = test.drop(columns=["SalePrice", "MachineID"])
y_test = test["SalePrice"]

# 2. Instanciation et entraînement de la baseline
dummy_median = DummyRegressor(strategy="median")
dummy_median.fit(X_train, y_train)

# 3. Prédictions
y_pred_train_dummy = dummy_median.predict(X_train)
y_pred_test_dummy = dummy_median.predict(X_test)

# 4. Évaluation (MAE)
mae_train_dummy = mean_absolute_error(y_train, y_pred_train_dummy)
mae_test_dummy = mean_absolute_error(y_test, y_pred_test_dummy)

print(f"Baseline MAE Train : {mae_train_dummy:.2f}")
print(f"Baseline MAE Test  : {mae_test_dummy:.2f}")

Baseline MAE Train : 16454.19
Baseline MAE Test  : 19697.33


Saledate est au format date hors les modèles que l'on souhaite utiliser n'accepte que des valeurs numériques. On va extraire l'année, le mois ,le trimestre 

In [3]:
X_train = extraire_dates(X_train, "saledate" )
X_test = extraire_dates(X_test, "saledate" )


On va encoder les variables non numérique afin de pouvoir tester nos différents modèles. Avant cela nous allons effectuer un dernier tri dans celles-ci. 

On supprime les variables redondantes ou encore celle qui sont utilisée comme des identifiants 


In [4]:
colonnes_a_supprimer = [
    'SalesID', 'ModelID', 'datasource', 'auctioneerID', 
    'fiModelDesc', 'fiBaseModel', 'fiSecondaryDesc', 
    'ProductGroupDesc', 'ProductSize'
]

X_train = X_train.drop(columns=colonnes_a_supprimer)
X_test = X_test.drop(columns=colonnes_a_supprimer)



In [5]:
colonnes_texte = X_train.select_dtypes(include=['object']).columns.tolist()
print(colonnes_texte)
print(f"Nombre de colonnes à encoder : {len(colonnes_texte)}")

['fiProductClassDesc', 'state', 'ProductGroup', 'Enclosure', 'Forks', 'Ride_Control', 'Transmission', 'Hydraulics', 'Coupler']
Nombre de colonnes à encoder : 9


C:\Users\Abram\AppData\Local\Temp\ipykernel_24220\1769016324.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_texte = X_train.select_dtypes(include=['object']).columns.tolist()


In [6]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

# Etape 8: Modélisation 

## 1 - Régression linéaire

### 1.1 - Encodage des variables 


## Diagnostic et traitement des valeurs manquantes

### Taux de NaN

| Colonne | NaN |
|---|---|
| `MachineHoursCurrentMeter` | 83 % |
| `Ride_Control` | 63 % |
| `Transmission` | 54 % |
| `Forks` | 52 % |
| `Coupler` | 47 % |
| `YearMade` | 9,5 % |
| `Enclosure` | 0,1 % |

Le traitement dépend de l'origine du manque, pas de son volume.

### Variables catégorielles

`Forks`, `Ride_Control`, `Transmission`, `Coupler` : NaN structurel, l'équipement n'existe pas sur ce type d'engin. L'information est déjà portée par `ProductGroup` et `fiProductClassDesc`. Imputation par la modalité `"Non applicable"`.

`Enclosure` : taux résiduel de 0,1 %, assimilable à une erreur de saisie. Imputation par `"Missing"`, modalité distincte puisque la nature du manque diffère.

L'imputation par le mode est écartée : elle attribuerait par exemple des `Forks` à un engin qui n'en possède structurellement pas.

### Variables numériques

Pour chaque groupe `fiProductClassDesc` :

- au moins 500 observations non-NaN sur la variable (train) : médiane du groupe
- sinon : médiane globale (train)

Le seuil de 500 est repris de l'étape 4 (regroupement des catégories rares), ce qui évite d'introduire un second paramètre arbitraire.

Le lissage bayésien vers la médiane globale a été écarté : la pondération linéaire par l'effectif n'est fondée que pour une moyenne, une moyenne pondérée de deux médianes n'étant pas la médiane de la population combinée. La moyenne n'est pas pour autant retenue comme substitut, `MachineHoursCurrentMeter` et `YearMade` restant asymétriques à droite après traitement des valeurs extrêmes (même argument qu'à l'étape 7 pour `SalePrice`).

### Fonction

`imputation_mediane_a_nan(train, test, colonne_groupement, colonne_imputer, seuil)` dans `preprocessing.py` : comptage des non-NaN par catégorie, sélection des catégories au-dessus du seuil, `.map()` vers leur médiane puis `.fillna()` sur la médiane globale pour les autres, application par `.fillna()` sur la colonne cible.

Toutes les médianes sont apprises sur train et réappliquées à test (anti-fuite).

Appliquée à `MachineHoursCurrentMeter` et `YearMade`. Vérification : 0 NaN restants sur `X_train` et `X_test`.

In [7]:
X_train, X_test = imputation_mediane_a_nan(X_train, X_test, "fiProductClassDesc", "MachineHoursCurrentMeter", 500)
X_train, X_test = imputation_mediane_a_nan(X_train, X_test, "fiProductClassDesc", "YearMade", 500)

print(X_train["MachineHoursCurrentMeter"].isna().sum())
print(X_train["YearMade"].isna().sum())
print(X_test["MachineHoursCurrentMeter"].isna().sum())
print(X_test["YearMade"].isna().sum())

0
0
0
0


In [8]:
X_train["Forks"] = X_train["Forks"].fillna("Non applicable")
X_train["Ride_Control"] = X_train["Ride_Control"].fillna("Non applicable")
X_train["Transmission"] = X_train["Transmission"].fillna("Non applicable")
X_train["Coupler"] = X_train["Coupler"].fillna("Non applicable")
X_train["Enclosure"] = X_train["Enclosure"].fillna("Missing")

X_test["Forks"] = X_test["Forks"].fillna("Non applicable")
X_test["Ride_Control"] = X_test["Ride_Control"].fillna("Non applicable")
X_test["Transmission"] = X_test["Transmission"].fillna("Non applicable")
X_test["Coupler"] = X_test["Coupler"].fillna("Non applicable")
X_test["Enclosure"] = X_test["Enclosure"].fillna("Missing")

print(X_train.isna().sum())
print(X_test.isna().sum())

YearMade                    0
MachineHoursCurrentMeter    0
fiProductClassDesc          0
state                       0
ProductGroup                0
Enclosure                   0
Forks                       0
Ride_Control                0
Transmission                0
Hydraulics                  0
Coupler                     0
hours_reported              0
saleyear                    0
salemonth                   0
salequarter                 0
dtype: int64
YearMade                    0
MachineHoursCurrentMeter    0
fiProductClassDesc          0
state                       0
ProductGroup                0
Enclosure                   0
Forks                       0
Ride_Control                0
Transmission                0
Hydraulics                  0
Coupler                     0
hours_reported              0
saleyear                    0
salemonth                   0
salequarter                 0
dtype: int64


## Encodage des variables catégorielles

Encodage adapté à la régression linéaire, première étape de la séquence de modélisation. Les arbres utiliseront un Ordinal Encoding, traité séparément.

### One-Hot

`ProductGroup`, `Transmission`, `Hydraulics`, `Forks`, `Ride_Control`, `Coupler` : pas d'ordre naturel exploitable et cardinalité limitée à quelques modalités, le coût en colonnes reste faible.

### Ordinal

`Enclosure` seule : l'ordre de confort est défendable (OROPS < EROPS < EROPS w AC).

`Coupler` avait été envisagée en Ordinal (Manual < Hydraulic), puis écartée. La modalité `"Non applicable"`, issue du traitement des NaN structurels, n'a pas de position cohérente sur cette échelle.

### Frequency

`fiProductClassDesc` (~61 modalités) et `state` (~50 modalités) : réduites chacune à une colonne numérique, sans recours à la cible. Le signal est indirect et modéré, mais le coût en dimensionnalité est nul.

Alternatives écartées pour `fiProductClassDesc` :

- extraction d'une variable numérique de tonnage ou de puissance : l'unité varie selon le type d'engin dans le texte brut (profondeur, puissance, poids), une extraction généralisable est trop complexe
- One-Hot : 61 coefficients supplémentaires, incompatible avec l'objectif d'interprétabilité de la régression linéaire
- Target Encoding : calculable sans fuite en apprenant sur train, mais la feature obtenue serait mécaniquement très corrélée à `SalePrice` et écraserait le poids des autres coefficients

`state` avait d'abord été classée en One-Hot, sur une confusion entre le lien de la variable avec l'engin et son coût en nombre de coefficients. Elle est reclassée par cohérence avec `fiProductClassDesc`, sur le même argument de dimensionnalité.

In [9]:
colonne_a_encoderOHE = ["ProductGroup", "Transmission", "Hydraulics", "Forks", "Ride_Control", "Coupler"]
encodeur_ohe = OneHotEncoder(drop="first", sparse_output=False)
resultat_ohe_train = encodeur_ohe.fit_transform(X_train[colonne_a_encoderOHE])
noms_colonnes_ohe_train= encodeur_ohe.get_feature_names_out()
df_ohe_train = pd.DataFrame(resultat_ohe_train, columns=noms_colonnes_ohe_train, index=X_train.index)
X_train = X_train.drop(columns=colonne_a_encoderOHE)
X_train = pd.concat([X_train, df_ohe_train], axis=1)
X_train.shape

(401125, 36)

In [10]:
resultat_ohe_test = encodeur_ohe.transform(X_test[colonne_a_encoderOHE])
noms_colonnes_ohe_test = encodeur_ohe.get_feature_names_out()
df_ohe_test = pd.DataFrame(resultat_ohe_test, columns=noms_colonnes_ohe_test, index=X_test.index)
X_test = X_test.drop(columns=colonne_a_encoderOHE)
X_test = pd.concat([X_test, df_ohe_test], axis=1)

In [11]:
X_test.shape

(11573, 36)

On impute le mode au données "missing" de enclosure. Au vu de la faible fréquence cela n'impactera pas les coefficients et permettra un encodage ordinal

In [12]:
mode_enclosure = X_train["Enclosure"].mode()[0]
X_train.loc[X_train["Enclosure"]=="Missing","Enclosure"] = mode_enclosure
X_train["Enclosure"].value_counts()

Enclosure
OROPS         174262
EROPS         139026
EROPS w AC     87837
Name: count, dtype: int64

In [13]:
mode_enclosure = X_train["Enclosure"].mode()[0]
X_test.loc[X_test["Enclosure"]=="Missing","Enclosure"] = mode_enclosure
X_test["Enclosure"].value_counts()

Enclosure
EROPS w AC    4782
OROPS         4048
EROPS         2743
Name: count, dtype: int64

In [14]:
encodeur_ordinal = OrdinalEncoder(categories=[["OROPS", "EROPS", "EROPS w AC"]])
resultat_ordinal_train = encodeur_ordinal.fit_transform(X_train[["Enclosure"]])
noms_colonnes_ordinal_train= encodeur_ordinal.get_feature_names_out()
df_ordinal_train = pd.DataFrame(resultat_ordinal_train, columns=noms_colonnes_ordinal_train, index=X_train.index)
X_train = X_train.drop(columns="Enclosure")
X_train = pd.concat([X_train, df_ordinal_train], axis=1)
X_train.shape

(401125, 36)

In [15]:
resultat_ordinal_test = encodeur_ordinal.transform(X_test[["Enclosure"]])
noms_colonnes_ordinal_test= encodeur_ordinal.get_feature_names_out()
df_ordinal_test= pd.DataFrame(resultat_ordinal_test, columns=noms_colonnes_ordinal_train, index=X_test.index)
X_test = X_test.drop(columns="Enclosure")
X_test = pd.concat([X_test, df_ordinal_test], axis=1)
X_test.shape

(11573, 36)

On applique le frequency encodage a state et fiProductClassDesc

In [16]:
Freq = X_train["fiProductClassDesc"].value_counts()
X_train["fiProductClassDesc_freq"] = X_train["fiProductClassDesc"].map(Freq)
X_train[["fiProductClassDesc", "fiProductClassDesc_freq"]].head(10)


,fiProductClassDesc,fiProductClassDesc_freq
0,Wheel Loader - 110.0 to 120.0 Horsepower,1041
1,Wheel Loader - 150.0 to 175.0 Horsepower,15114
2,Skid Steer Loader - 1351.0 to 1601.0 Lb Operat...,9321
3,"Hydraulic Excavator, Track - 12.0 to 14.0 Metr...",11354
4,Skid Steer Loader - 1601.0 to 1751.0 Lb Operat...,9011
5,Backhoe Loader - 14.0 to 15.0 Ft Standard Digg...,56166
6,"Hydraulic Excavator, Track - 21.0 to 24.0 Metr...",13323
7,Backhoe Loader - 14.0 to 15.0 Ft Standard Digg...,56166
8,"Hydraulic Excavator, Track - 3.0 to 4.0 Metric...",5860
9,Wheel Loader - 350.0 to 500.0 Horsepower,3024


In [17]:
Freq = X_train["fiProductClassDesc"].value_counts()
X_test["fiProductClassDesc_freq"] = X_test["fiProductClassDesc"].map(Freq)
X_test[["fiProductClassDesc", "fiProductClassDesc_freq"]].head(100)

,fiProductClassDesc,fiProductClassDesc_freq
401125,Other,2344
401126,"Hydraulic Excavator, Track - 28.0 to 33.0 Metr...",6134
401127,"Hydraulic Excavator, Track - 24.0 to 28.0 Metr...",6235
401128,"Hydraulic Excavator, Track - 28.0 to 33.0 Metr...",6134
401129,Wheel Loader - 120.0 to 135.0 Horsepower,10551
...,...,...
401220,Backhoe Loader - 14.0 to 15.0 Ft Standard Digg...,56166
401221,Skid Steer Loader - 1601.0 to 1751.0 Lb Operat...,9011
401222,Skid Steer Loader - 1351.0 to 1601.0 Lb Operat...,9321
401223,Backhoe Loader - 14.0 to 15.0 Ft Standard Digg...,56166


In [18]:
Freq = X_train["state"].value_counts()
X_train["state_freq"] = X_train["state"].map(Freq)
X_train[["state", "state_freq"]].head(10)

,state,state_freq
0,Alabama,9997
1,North Carolina,10404
2,New York,8604
3,Texas,51682
4,New York,8604
5,Arizona,9173
6,Florida,63944
7,Illinois,11209
8,Texas,51682
9,Florida,63944


In [19]:
Freq = X_train["state"].value_counts()
X_test["state_freq"] = X_test["state"].map(Freq)
X_test[["state", "state_freq"]].head(10)

,state,state_freq
401125,Kentucky,5278
401126,Connecticut,8128
401127,Connecticut,8128
401128,Connecticut,8128
401129,Florida,63944
401130,Florida,63944
401131,Illinois,11209
401132,Illinois,11209
401133,Florida,63944
401134,West Virginia,746


In [20]:
Colonne_a_drop=["state","fiProductClassDesc"]
X_train = X_train.drop(columns=Colonne_a_drop)
X_test= X_test.drop(columns=Colonne_a_drop)

In [21]:
X_train.shape
X_test.shape
X_train.columns.tolist()

['YearMade',
 'MachineHoursCurrentMeter',
 'hours_reported',
 'saleyear',
 'salemonth',
 'salequarter',
 'ProductGroup_MG',
 'ProductGroup_SSL',
 'ProductGroup_TEX',
 'ProductGroup_TTT',
 'ProductGroup_WL',
 'Transmission_Autoshift',
 'Transmission_Direct Drive',
 'Transmission_Hydrostatic',
 'Transmission_Non applicable',
 'Transmission_None or Unspecified',
 'Transmission_Powershift',
 'Transmission_Powershuttle',
 'Transmission_Standard',
 'Hydraulics_3 Valve',
 'Hydraulics_4 Valve',
 'Hydraulics_Auxiliary',
 'Hydraulics_Base + 1 Function',
 'Hydraulics_Other',
 'Hydraulics_Standard',
 'Forks_None or Unspecified',
 'Forks_Yes',
 'Ride_Control_Non applicable',
 'Ride_Control_None or Unspecified',
 'Ride_Control_Yes',
 'Coupler_Manual',
 'Coupler_Non applicable',
 'Coupler_None or Unspecified',
 'Enclosure',
 'fiProductClassDesc_freq',
 'state_freq']

In [22]:
X_train["Enclosure"].dtype

dtype('float64')

In [23]:
X_train.columns.tolist()

['YearMade',
 'MachineHoursCurrentMeter',
 'hours_reported',
 'saleyear',
 'salemonth',
 'salequarter',
 'ProductGroup_MG',
 'ProductGroup_SSL',
 'ProductGroup_TEX',
 'ProductGroup_TTT',
 'ProductGroup_WL',
 'Transmission_Autoshift',
 'Transmission_Direct Drive',
 'Transmission_Hydrostatic',
 'Transmission_Non applicable',
 'Transmission_None or Unspecified',
 'Transmission_Powershift',
 'Transmission_Powershuttle',
 'Transmission_Standard',
 'Hydraulics_3 Valve',
 'Hydraulics_4 Valve',
 'Hydraulics_Auxiliary',
 'Hydraulics_Base + 1 Function',
 'Hydraulics_Other',
 'Hydraulics_Standard',
 'Forks_None or Unspecified',
 'Forks_Yes',
 'Ride_Control_Non applicable',
 'Ride_Control_None or Unspecified',
 'Ride_Control_Yes',
 'Coupler_Manual',
 'Coupler_Non applicable',
 'Coupler_None or Unspecified',
 'Enclosure',
 'fiProductClassDesc_freq',
 'state_freq']

On va standardiser les colonnes

In [26]:
Colonne_a_standardiser = ["YearMade", "MachineHoursCurrentMeter", "saleyear", "salemonth", "salequarter", "fiProductClassDesc_freq", "state_freq"]
standardisation = StandardScaler()
Resultat_standardisation =standardisation.fit_transform(X_train[Colonne_a_standardiser])
Resultat_standardisation.shape
X_train[Colonne_a_standardiser] = Resultat_standardisation
Resultat_standardisation =standardisation.transform(X_test[Colonne_a_standardiser])
X_test[Colonne_a_standardiser] = Resultat_standardisation

In [27]:
X_train[Colonne_a_standardiser].describe()

,YearMade,MachineHoursCurrentMeter,saleyear,salemonth,salequarter,fiProductClassDesc_freq,state_freq
count,4.011250e+05,4.011250e+05,4.011250e+05,4.011250e+05,4.011250e+05,4.011250e+05,4.011250e+05
mean,1.026661e-14,-4.988192e-17,-1.723193e-14,8.842703e-17,1.473784e-17,-2.720832e-17,8.389231e-17
std,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00
min,-7.920263e+00,-1.413666e+00,-2.623435e+00,-1.578894e+00,-1.217516e+00,-8.532605e-01,-1.112318e+00
25%,-6.236146e-01,-5.207369e-01,-7.117827e-01,-9.948794e-01,-1.217516e+00,-5.417481e-01,-7.028246e-01
50%,2.348146e-01,-3.529704e-01,3.309369e-01,-1.188573e-01,-3.385489e-01,-3.740377e-01,-5.336871e-01
75%,7.713329e-01,3.898244e-01,8.522968e-01,7.571648e-01,5.404182e-01,-2.064179e-02,1.195239e+00
max,1.951673e+00,1.220110e+01,1.199870e+00,1.633187e+00,1.419385e+00,2.413361e+00,1.742747e+00
